# Spam Email Classifier — SageMaker Training

Train a text classifier on `spam.csv` to detect spam emails/messages.

**Expected CSV format** (one of):
- `label,text` — labels like `spam` / `ham`
- `Category,Message` — common Kaggle format
- `v1,v2` — classic SMS Spam Collection format

**If you get Missing file:** upload `spam.csv`, then set `DATA_PATH` or `S3_URI` in Setup.

**SDK note:** use SageMaker Python SDK **v2** (`sagemaker>=2.232,<3`).


## 1. Setup

In [ ]:
# Pin sagemaker to v2 — v3 removed sagemaker.sklearn.SKLearn
%pip install -q --upgrade "scikit-learn" "pandas" "joblib" "boto3" "sagemaker>=2.232.0,<3"

In [ ]:
import os
import json
import joblib
from pathlib import Path

import boto3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

# --- Config ---
DATA_PATH = Path("spam.csv")  # change if your file is elsewhere
OUTPUT_DIR = Path("model_output")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

print("Ready. Data path:", DATA_PATH.resolve())

## 2. Load and normalize data

In [ ]:
def load_spam_csv(path: Path) -> pd.DataFrame:
    """Load spam.csv and normalize to columns: label (0/1), text."""
    # Try common encodings used by public spam datasets
    for encoding in ("utf-8", "latin-1", "iso-8859-1", "cp1252"):
        try:
            df = pd.read_csv(path, encoding=encoding)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"Could not decode {path}")

    # Drop empty unnamed columns (common in SMS spam CSV)
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]

    col_map = {c.lower().strip(): c for c in df.columns}

    # Detect label + text columns
    label_col = None
    text_col = None

    for candidate in ("label", "category", "class", "v1", "spam"):
        if candidate in col_map:
            label_col = col_map[candidate]
            break

    for candidate in ("text", "message", "email", "content", "body", "v2"):
        if candidate in col_map:
            text_col = col_map[candidate]
            break

    if label_col is None or text_col is None:
        if df.shape[1] >= 2:
            label_col, text_col = df.columns[0], df.columns[1]
            print(f"Using first two columns: {label_col!r}, {text_col!r}")
        else:
            raise ValueError(
                f"Could not find label/text columns. Columns={list(df.columns)}"
            )

    out = pd.DataFrame(
        {
            "label_raw": df[label_col].astype(str).str.strip().str.lower(),
            "text": df[text_col].astype(str).str.strip(),
        }
    )
    out = out[out["text"].str.len() > 0].drop_duplicates()

    # Map labels → 1 = spam, 0 = ham/not spam
    spam_values = {"spam", "1", "true", "yes"}
    ham_values = {"ham", "not spam", "legit", "0", "false", "no"}

    def to_binary(v: str) -> int:
        if v in spam_values:
            return 1
        if v in ham_values:
            return 0
        raise ValueError(f"Unknown label value: {v!r}")

    out["label"] = out["label_raw"].map(to_binary)
    return out[["label", "text"]].reset_index(drop=True)


data_path = Path(DATA_PATH)
if not data_path.is_file():
    raise FileNotFoundError(
        f"Missing file: {data_path}\n\n"
        "Upload spam.csv, then either:\n"
        "  1) Put it in the notebook folder / shared Space, and re-run Setup\n"
        "  2) Or set S3_URI = 's3://bucket/path/spam.csv' in the Setup cell\n"
        "  3) Or set DATA_PATH = Path('/full/path/to/spam.csv')"
    )

data = load_spam_csv(data_path)
print(data.head())
print("\nShape:", data.shape)
print("Label distribution:")
print(data["label"].value_counts().rename({0: "ham", 1: "spam"}))

## 3. Train / test split

In [ ]:
X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print("Train spam rate:", round(y_train.mean(), 3))
print("Test spam rate:", round(y_test.mean(), 3))

## 4. Build and train model

Pipeline: **TF-IDF** → **Logistic Regression** (strong baseline for spam detection).

In [ ]:
pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                min_df=2,
                max_features=20000,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

pipeline.fit(X_train, y_train)
print("Training complete.")

## 5. Evaluate

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"F1 (spam): {f1:.4f}")
print("\\nClassification report:")
print(
    classification_report(
        y_test, y_pred, target_names=["ham", "spam"], digits=4
    )
)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred))

## 6. Quick inference examples

In [ ]:
def predict_spam(texts, threshold=0.5):
    if isinstance(texts, str):
        texts = [texts]
    probs = pipeline.predict_proba(texts)[:, 1]
    rows = []
    for text, p in zip(texts, probs):
        rows.append(
            {
                "text": text[:80] + ("…" if len(text) > 80 else ""),
                "spam_probability": round(float(p), 4),
                "prediction": "spam" if p >= threshold else "ham",
            }
        )
    return pd.DataFrame(rows)


examples = [
    "Hi, can we move our meeting to tomorrow morning?",
    "CONGRATULATIONS! You won a FREE iPhone. Click here NOW to claim your prize!!!",
    "Your Amazon order #12345 has shipped and will arrive on Friday.",
    "URGENT: Your account will be closed. Verify password at http://bit.ly/fake",
]

predict_spam(examples)

## 7. Save model artifacts

In [ ]:
model_path = OUTPUT_DIR / "spam_model.joblib"
metrics_path = OUTPUT_DIR / "metrics.json"

joblib.dump(pipeline, model_path)

metrics = {
    "accuracy": float(acc),
    "f1_spam": float(f1),
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "spam_rate_train": float(y_train.mean()),
}
metrics_path.write_text(json.dumps(metrics, indent=2))

print("Saved:", model_path.resolve())
print("Saved:", metrics_path.resolve())
print(json.dumps(metrics, indent=2))

## 8. (Optional) Upload artifacts to S3

Run this if you want to store the model in your AWS account from SageMaker Studio / notebook.

In [ ]:
UPLOAD_TO_S3 = False  # set True when ready

BUCKET = os.environ.get("SM_BUCKET") or "YOUR-BUCKET-NAME"
PREFIX = "spam-classifier/model"

if UPLOAD_TO_S3:
    session = boto3.Session()
    region = session.region_name or "us-east-1"
    s3 = session.client("s3")

    for path in (model_path, metrics_path):
        key = f"{PREFIX}/{path.name}"
        s3.upload_file(str(path), BUCKET, key)
        print(f"s3://{BUCKET}/{key}")
else:
    print("Skipped S3 upload. Set UPLOAD_TO_S3 = True and BUCKET to enable.")

## 9. (Optional) SageMaker Training Job

Use this when you want a managed training job instead of training inside the notebook.

1. Upload `spam.csv` to S3
2. Create `train.py` (script mode)
3. Launch `SKLearn` estimator

In [ ]:
RUN_SAGEMAKER_JOB = False  # set True to launch a training job

if RUN_SAGEMAKER_JOB:
    import importlib
    import sagemaker

    # SKLearn estimator exists only in SageMaker Python SDK v2
    major = int(sagemaker.__version__.split(".")[0])
    if major >= 3:
        raise ImportError(
            f"Installed sagemaker=={sagemaker.__version__} (v3). "
            "SKLearn was removed in v3.\n"
            "Fix: run the install cell again, then restart the kernel:\n"
            '  %pip install -q --upgrade "sagemaker>=2.232.0,<3"'
        )

    try:
        from sagemaker.sklearn.estimator import SKLearn
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError(
            "Cannot import sagemaker.sklearn. "
            "Re-run the install cell and Restart Kernel, then try again.\n"
            f"Current sagemaker version: {sagemaker.__version__}"
        ) from e

    print("sagemaker", sagemaker.__version__)
    sm_session = sagemaker.Session()
    role = sagemaker.get_execution_role()
    bucket = sm_session.default_bucket()
    prefix = "spam-classifier"

    # Upload training data
    train_s3 = sm_session.upload_data(
        path=str(DATA_PATH),
        bucket=bucket,
        key_prefix=f"{prefix}/data",
    )
    print("Training data:", train_s3)

    # Write a minimal training script for script mode
    script_dir = Path("sagemaker_script")
    script_dir.mkdir(exist_ok=True)
    (script_dir / "train.py").write_text(
        '''
import argparse
import json
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score


def load_data(path):
    for enc in ("utf-8", "latin-1", "iso-8859-1"):
        try:
            df = pd.read_csv(path, encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
    cols = {c.lower(): c for c in df.columns}
    label_col = next((cols[k] for k in ("label", "category", "v1") if k in cols), df.columns[0])
    text_col = next((cols[k] for k in ("text", "message", "v2") if k in cols), df.columns[1])
    y = df[label_col].astype(str).str.lower().str.strip().map(lambda v: 1 if v in {"spam", "1"} else 0)
    X = df[text_col].astype(str)
    return X, y


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    args = parser.parse_args()

    csv_path = args.train
    if os.path.isdir(csv_path):
        files = [f for f in os.listdir(csv_path) if f.endswith(".csv")]
        csv_path = os.path.join(csv_path, files[0])

    X, y = load_data(csv_path)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=20000)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1_spam": float(f1_score(y_test, pred)),
    }
    print("metrics:", metrics)

    joblib.dump(pipe, os.path.join(args.model_dir, "model.joblib"))
    with open(os.path.join(args.model_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f)
'''.lstrip()
    )

    estimator = SKLearn(
        entry_point="train.py",
        source_dir=str(script_dir),
        role=role,
        instance_count=1,
        instance_type="ml.m5.large",
        framework_version="1.2-1",
        py_version="py3",
        hyperparameters={},
    )

    estimator.fit({"train": train_s3})
    print("Training job finished.")
    print("Model artifact:", estimator.model_data)
else:
    print("Skipped SageMaker Training Job. Set RUN_SAGEMAKER_JOB = True to launch.")

## Done

- Local training + evaluation: sections 1–7
- Optional S3 upload: section 8
- Optional managed SageMaker Training Job: section 9

**Next steps (outside this notebook):** deploy with `SKLearnModel` + endpoint, or load `model_output/spam_model.joblib` for batch scoring.
